In [ ]:
import pandas as pd 

original = pd.read_csv("insert original fitzpatrick.csv here")
original

In [ ]:
new = original.copy().drop(columns=["label", "url", "url_alphanum"])#, "qc", "fitzpatrick_centaur", "nine_partition_label"])
new


In [ ]:
renamed = new.rename(columns={"fitzpatrick_scale":"skin_tone", 
                        "fitzpatrick_centaur":"alternative_skin_tone", 
                        "nine_partition_label":"super_label", "three_partition_label":"label", "qc":"expert_opinion"})

#renamed = new.rename(columns={"fitzpatrick_scale":"skin_tone", "fitzpatrick_centaur":"alternative_skin_tone", 
 #                               "nine_partition_label":"additional_info", "three_partition_label":"label"})
renamed

In [ ]:
renamed.columns

In [ ]:
column_shuffled = renamed[['md5hash', 'super_label','skin_tone', 'alternative_skin_tone', 'expert_opinion', 'label']]
#column_shuffled = renamed[['label','md5hash','skin_tone']]
column_shuffled

In [ ]:
shuffled = column_shuffled.sample(frac=1, random_state=4242).reset_index(drop=True)
shuffled

In [ ]:
len(shuffled)

In [ ]:
mapping_df = pd.DataFrame({"md5hash":shuffled["md5hash"], "image_name":[f"image_{i}" for i in range (1, len(shuffled) +1)]})

mapping_dict= dict(zip(mapping_df["md5hash"], mapping_df["image_name"]))


mapping_df.to_csv("mapping_dict.csv")

In [ ]:
import os
import shutil

source_dir = "insert downloaded Fitzpatrick17k images here"
target_dir = "./MyData/MyImages"

os.makedirs(target_dir, exist_ok=True)

copied = 0
missing = []

for _, row in mapping_df.iterrows():
    src = os.path.join(source_dir, f"{row['md5hash']}.jpg")
    dst = os.path.join(target_dir, f"{row['image_name']}.jpg")

    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(f"{row['md5hash']}.jpg")

print(f"Copied {copied} images to {target_dir}")

if missing:
    print(f"Missing {len(missing)} files")
    print(missing[:20])

In [ ]:
image_renamed = shuffled.copy()
image_renamed["image_name"] = image_renamed["md5hash"].map(mapping_dict)
image_renamed = image_renamed.drop(columns="md5hash")
image_renamed

In [ ]:
#non_neoplastic_renamed = image_renamed.copy()
#non_neoplastic_renamed["label"] = non_neoplastic_renamed.label.replace({"non-neoplastic":"inflammatory"})
#non_neoplastic_renamed

In [ ]:
image_renamed.to_csv("./MyData/mydataset.csv")